In [12]:
# Core libraries
import pandas as pd
import numpy as np
import json

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical analysis
from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Project config
from src.params import *
from src.ingestion.openaq import OpenAQClient

# Plot configuration
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [13]:
client = OpenAQClient(api_key= API_AQ, radius= 7000)

In [14]:
rio = client.fetch_location(lat= -22.9083, lon= -43.1764)

In [20]:
rio

{'meta': {'name': 'openaq-api',
  'website': '/',
  'page': 1,
  'limit': 1000,
  'found': 8},
 'results': [{'id': 820323,
   'name': 'Centro Kunak ',
   'locality': 'Rio de Janeiro',
   'timezone': 'America/Sao_Paulo',
   'country': {'id': 45, 'code': 'BR', 'name': 'Brazil'},
   'owner': {'id': 4, 'name': 'Unknown Governmental Organization'},
   'provider': {'id': 62, 'name': 'Rio City Hall'},
   'isMobile': False,
   'isMonitor': True,
   'instruments': [{'id': 2, 'name': 'Government Monitor'}],
   'sensors': [{'id': 4475452,
     'name': 'co ppm',
     'parameter': {'id': 8,
      'name': 'co',
      'units': 'ppm',
      'displayName': 'CO'}},
    {'id': 4475460,
     'name': 'o3 µg/m³',
     'parameter': {'id': 3,
      'name': 'o3',
      'units': 'µg/m³',
      'displayName': 'O₃ mass'}},
    {'id': 6344694,
     'name': 'pm10 µg/m³',
     'parameter': {'id': 1,
      'name': 'pm10',
      'units': 'µg/m³',
      'displayName': 'PM10'}},
    {'id': 14798912,
     'name': 'pm25 µ

In [18]:
rio_sensors = client.filter_sensors(rio, start_project_date= START_PROJECT_DATE_STR, end_project_date= END_PROJECT_DATE_STR)
rio_sensors

Initial monitor-grade sensors: 8
Sensors active at end date: 8
Sensors with full coverage: 1


[14798912]

In [19]:
rio_data = client.fetch_one_sensor_data(sensor_id=14798912, start_date= START_TRAIN_DATE_STR, end_date= END_TRAIN_DATE_STR, cache_dir="../data/cache/Rio de Janeiro")

In [3]:
df_aq = pd.read_csv("../data/raw/aq_data.csv")

In [11]:
df_aq["city"].unique()

array(['Paris', 'Lyon', 'New York', 'Los Angeles', 'Delhi', 'London',
       'Berlin', 'Rome', 'Barcelona', 'Mumbai', 'Mexico City', 'Santiago'],
      dtype=object)

# Base EDA

## Stats on sensors availability

In [ ]:
sensors_stats = pd.DataFrame(df_aq.groupby("city")["sensor_id"].nunique())


,sensor_id
city,
Barcelona,1
Berlin,3
Delhi,10
London,14
Los Angeles,1
Lyon,3
Mexico City,3
Mumbai,5
New York,6


In [ ]:
# cities and sensors and average daily coverage
stats_sensors = pd.DataFrame({"cities": df_aq["city"].unique(),
 "nb_sensors": df_aq.groupby("city")["sensor_id"].nunique()
 })


,cities,nb_sensors
city,,
Barcelona,Paris,1
Berlin,Lyon,3
Delhi,New York,10
London,Los Angeles,14
Los Angeles,Delhi,1
Lyon,London,3
Mexico City,Berlin,3
Mumbai,Rome,5
New York,Barcelona,6
